# Early-Exit Tuning + Benchmark (LLaMA-3-8B)

This notebook focuses on training and benchmark only.

Flow: **Mount Drive -> Install -> Train EE heads -> Push to HF Hub -> Benchmark from HF heads -> Download JSON results**

Benchmark metrics: **TTFT**, **per-token latency**, **end-to-end latency**, and **tokens/joule**.

## 1. Setup

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Clone the repo (or upload manually)
!git clone https://github.com/airlanggawicaksono/ls-del-training.git /content/ls-del-training
%cd /content/ls-del-training

In [ ]:
!pip install -q -r requirements.txt
# Fix for C4 loading issues from HF discussion: keep these two libs up to date
!pip install -q -U datasets huggingface_hub

In [ ]:
import datasets
import huggingface_hub

print("datasets:", datasets.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

In [ ]:
# Hugging Face login (required for gated model + push/pull from Hub)
# Set HF_TOKEN in Colab Secrets (key icon, left sidebar)
from huggingface_hub import login, whoami
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN, add_to_git_credential=False)
print("Logged in as:", whoami().get("name", "unknown"))

In [ ]:
import torch

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 2. Train Config

In [ ]:
import os

# ---- Paths / Repo ----
DRIVE_OUTPUT = "/content/drive/MyDrive/ee-llama3-8b"
# Use your exact HF repo id for exit heads
HF_REPO = "wicaksonolxn/llama3-8b-ee-heads"

# ---- Base model for both EE and baseline benchmark ----
MODEL = "meta-llama/Meta-Llama-3-8B"

# ---- Early-exit settings ----
EXIT_LAYERS = [8, 16, 24]
EXIT_WEIGHTS = [0.4, 0.6, 0.8]
CONFIDENCE_THRESHOLD = 0.9

# ---- Training size ----
# Start smaller first for faster iteration; increase once pipeline is stable.
MAX_TRAIN_SAMPLES = 200_000
MAX_VAL_SAMPLES = 2_000

# ---- Train hyperparams ----
BATCH_SIZE = 1
GRAD_ACCUM = 16
LR = 1e-4
EPOCHS = 1
SEQ_LEN = 2048
DTYPE = "bfloat16"

# ---- Optional local C4 cache on Drive ----
USE_LOCAL_C4_CACHE = True
C4_CACHE_DIR = "/content/drive/MyDrive/c4_cache"
C4_TRAIN_FILE = os.path.join(C4_CACHE_DIR, "c4_train.jsonl")
C4_VAL_FILE = os.path.join(C4_CACHE_DIR, "c4_validation.jsonl")
C4_PREDOWNLOAD_TRAIN_SAMPLES = MAX_TRAIN_SAMPLES
C4_PREDOWNLOAD_VAL_SAMPLES = MAX_VAL_SAMPLES
FORCE_REDOWNLOAD_C4 = False

# ---- Benchmark settings ----
BENCHMARK_SAMPLES = 100
BENCHMARK_MAX_NEW_TOKENS = 128
RESULTS_PATH = os.path.join(DRIVE_OUTPUT, "benchmark_results_cnn_dm_100.json")

os.makedirs(DRIVE_OUTPUT, exist_ok=True)
os.makedirs(C4_CACHE_DIR, exist_ok=True)

## 2.6 Optional: Clear Only C4 Cache (if cache/config mismatch errors)

Use only if you hit C4 cache errors such as `multiple configurations in cache` or `Couldn't find cache for allenai/c4 ...`.

In [ ]:
import shutil
import os

c4_cache = os.path.expanduser("~/.cache/huggingface/datasets/allenai___c4")
if os.path.exists(c4_cache):
    shutil.rmtree(c4_cache)
    print("Deleted:", c4_cache)
else:
    print("C4 cache not found, nothing to delete.")

## 2.5 Optional: Pre-Download C4 Subset To Drive

Run once to cache C4 subset on Drive. Later runs can train from local JSONL (`train_file` / `validation_file`) and skip repeated remote dataset pulls.

In [ ]:
import json
from datasets import load_dataset


def _stream_to_jsonl(split_name: str, n_samples: int, out_path: str):
    print(f"Caching {split_name} -> {out_path} ({n_samples} rows)")
    ds = load_dataset("allenai/c4", "en", split=split_name, streaming=True)
    written = 0
    with open(out_path, "w", encoding="utf-8") as f:
        for row in ds:
            text = row.get("text", "")
            f.write(json.dumps({"text": text}, ensure_ascii=False) + "\n")
            written += 1
            if written >= n_samples:
                break
    print(f"Wrote {written:,} rows to {out_path}")


if FORCE_REDOWNLOAD_C4 or not os.path.exists(C4_TRAIN_FILE):
    _stream_to_jsonl("train", C4_PREDOWNLOAD_TRAIN_SAMPLES, C4_TRAIN_FILE)
else:
    print("Train cache exists, skipping:", C4_TRAIN_FILE)

if FORCE_REDOWNLOAD_C4 or not os.path.exists(C4_VAL_FILE):
    _stream_to_jsonl("validation", C4_PREDOWNLOAD_VAL_SAMPLES, C4_VAL_FILE)
else:
    print("Validation cache exists, skipping:", C4_VAL_FILE)

In [ ]:
# Write config consumed by finetune_ee.py
exit_layers_str = "[" + ", ".join(str(x) for x in EXIT_LAYERS) + "]"
exit_weights_str = "[" + ", ".join(str(x) for x in EXIT_WEIGHTS) + "]"

if USE_LOCAL_C4_CACHE and os.path.exists(C4_TRAIN_FILE) and os.path.exists(C4_VAL_FILE):
    dataset_block = f"""train_file = {C4_TRAIN_FILE}
validation_file = {C4_VAL_FILE}
text_column = text"""
    print("Using local cached C4 files from Drive.")
else:
    dataset_block = """dataset_name = allenai/c4
dataset_config_name = en
text_column = text"""
    print("Using remote HF C4 dataset directly.")

config_text = f"""model_name_or_path = {MODEL}
{dataset_block}
output_dir = {DRIVE_OUTPUT}

max_train_samples = {MAX_TRAIN_SAMPLES}
max_val_samples = {MAX_VAL_SAMPLES}

max_seq_length = {SEQ_LEN}
per_device_train_batch_size = {BATCH_SIZE}
per_device_eval_batch_size = {BATCH_SIZE}
gradient_accumulation_steps = {GRAD_ACCUM}
learning_rate = {LR}
num_train_epochs = {EPOCHS}

logging_steps = 10
save_steps = 500
eval_steps = 500
seed = 42

torch_dtype = {DTYPE}
padding_side = right
report_to = [\"none\"]
overwrite_output_dir = true

exit_layer_indices = {exit_layers_str}
exit_loss_weights = {exit_weights_str}
init_exit_from_base = true
exit_confidence_threshold = {CONFIDENCE_THRESHOLD}
hub_exit_heads_repo = {HF_REPO}
"""

config_path = "/content/ee_config"
with open(config_path, "w", encoding="utf-8") as f:
    f.write(config_text)

print(config_text)

## 3. Train (Push Exit Heads + Training Logs to HF Hub)

`finetune_ee.py` will train exit heads, save artifacts to Drive, push **exit heads** to hub_exit_heads_repo, and push **training logs** to logs/train in the same HF repo.

In [ ]:
!python finetune_ee.py --config /content/ee_config

In [ ]:
# Optional: verify pushed files exist on HF Hub
from huggingface_hub import list_repo_files

files = list_repo_files(HF_REPO)
print("HF repo files:")
for f in files:
    print(" -", f)

## 4. Benchmark from HF Heads (Baseline = Original LLaMA)

This runs on CNN/DailyMail (`BENCHMARK_SAMPLES`, default 100).
Both EE and baseline are compiled inside `run_full_benchmark()` for fair comparison.

In [ ]:
import torch
from ee.benchmark import run_full_benchmark

dtype = torch.bfloat16 if DTYPE == "bfloat16" else torch.float16

results = run_full_benchmark(
    base_model_name=MODEL,
    exit_heads_repo_or_dir=HF_REPO,
    exit_layer_indices=EXIT_LAYERS,
    n_samples=BENCHMARK_SAMPLES,
    max_new_tokens=BENCHMARK_MAX_NEW_TOKENS,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    torch_dtype=dtype,
    output_path=RESULTS_PATH,
    push_results_to_hub_repo=HF_REPO,
    push_results_path_in_repo="logs/benchmark/benchmark_results_cnn_dm_100.json",
)

print("Saved benchmark JSON to:", RESULTS_PATH)
print(
    "Pushed benchmark JSON to HF repo path: logs/benchmark/benchmark_results_cnn_dm_100.json"
)

In [ ]:
# Quick view of the latency/energy section
print("EE latency/energy:", results["ee_latency"])
print("Baseline latency/energy:", results["baseline_latency"])

## 5. Download Benchmark JSON

In [ ]:
from google.colab import files

files.download(RESULTS_PATH)